# IF merge

Lecture、Exercise、Exam 分开合并；最后一个 cell 单独翻译所有合并后的 PDF。

In [1]:
from pathlib import Path
from collections import defaultdict
import html
import re

from pypdf import PdfReader, PdfWriter

IF_DIR = Path(r"E:\OneDrive - MSFT\.master_data\26ss\IF")
LECTURE_DIR = IF_DIR / "Vorlesungsfolien"
EXERCISE_SLIDES_DIR = IF_DIR / "Uebungsblaetter - Folien"
EXERCISE_TASKS_DIR = IF_DIR / "Uebungsblaetter"
EXERCISE_SOLUTIONS_DIR = IF_DIR / "Uebungsblaetter - Loesungen"
EXAM_DIR = IF_DIR / "Weiteres Lernmaterial" / "Altklausuren_IF_bis_ss25"

LECTURE_OUTPUT_DIR = LECTURE_DIR / "_merged"
EXERCISE_OUTPUT_DIR = IF_DIR / "Uebungsblaetter - merged"
EXAM_OUTPUT_DIR = EXAM_DIR / "_merged"

def pdf_pages(path):
    return len(PdfReader(str(path)).pages)

def merge_pdf_files(files, output_path, outline_titles=None):
    writer = PdfWriter()
    file_pages = []
    for index, path in enumerate(files):
        pages = pdf_pages(path)
        title = outline_titles[index] if outline_titles else None
        writer.append(str(path), outline_item=title, import_outline=False)
        file_pages.append((path, pages))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as output:
        writer.write(output)
    return file_pages, len(writer.pages)


## 1. Merge lecture material by chapter

In [2]:
lecture_groups = defaultdict(list)
chapter_titles = {}
for path in LECTURE_DIR.glob("*.pdf"):
    match = re.match(r"^(\d+)\.\s*(.+)$", path.stem)
    if not match:
        continue
    chapter = int(match.group(1))
    title = html.unescape(match.group(2)).replace("/", "-")
    lecture_groups[chapter].append(path)
    chapter_titles[chapter] = title

LECTURE_OUTPUT_DIR.mkdir(exist_ok=True)
for chapter in sorted(lecture_groups):
    title = chapter_titles[chapter]
    file_title = re.sub(r"\s+", "_", title)
    output_path = LECTURE_OUTPUT_DIR / f"Chapter{chapter:02d}_{file_title}.pdf"
    files = sorted(lecture_groups[chapter], key=lambda path: path.name)
    pages, total = merge_pdf_files(files, output_path)
    print(f"{output_path.name}: {len(pages)} files, {total} pages")


Chapter01_Einführung.pdf: 1 files, 85 pages
Chapter02_Struktur_von_Fusionssystemen.pdf: 1 files, 49 pages
Chapter03_Probabilistische_Methoden_-_Klassische_Methoden.pdf: 1 files, 81 pages
Chapter04_Probabilistische_Methoden_-_Bayes'sche_Methoden.pdf: 1 files, 155 pages
Chapter05_Dempster-Shafer-Theorie.pdf: 1 files, 30 pages
Chapter06_Fuzzy-Systeme.pdf: 1 files, 38 pages
Chapter08_Registrierung.pdf: 1 files, 32 pages
Chapter09_Energiefunktionale.pdf: 1 files, 17 pages


In [3]:
# 将全部 Lecture 合并为一个 PDF，并为每个 Chapter 添加一级目录。
def lecture_sort_key(path):
    match = re.match(r"^Chapter(\d+)", path.stem, re.IGNORECASE)
    return int(match.group(1)) if match else 9999

def lecture_outline_title(path):
    match = re.match(r"^(Chapter\d+)_(.+)$", path.stem, re.IGNORECASE)
    return f"{match.group(1)}: {match.group(2).replace('_', ' ')}"

chapter_files = sorted([
    path for path in LECTURE_OUTPUT_DIR.glob("Chapter*.pdf")
    if not path.name.startswith("trans-")
], key=lecture_sort_key)
if not chapter_files:
    raise FileNotFoundError(f"没有找到 Chapter PDF：{LECTURE_OUTPUT_DIR}")

all_lectures_output = LECTURE_OUTPUT_DIR / "IF_All_Chapters.pdf"
titles = [lecture_outline_title(path) for path in chapter_files]
pages, total = merge_pdf_files(chapter_files, all_lectures_output, titles)
for path, count in pages:
    print(f"{path.name} -> {count} pages")
print(f"合并完成：{all_lectures_output}，总页数：{total}")


Chapter01_Einführung.pdf -> 85 pages
Chapter02_Struktur_von_Fusionssystemen.pdf -> 49 pages
Chapter03_Probabilistische_Methoden_-_Klassische_Methoden.pdf -> 81 pages
Chapter04_Probabilistische_Methoden_-_Bayes'sche_Methoden.pdf -> 155 pages
Chapter05_Dempster-Shafer-Theorie.pdf -> 30 pages
Chapter06_Fuzzy-Systeme.pdf -> 38 pages
Chapter08_Registrierung.pdf -> 32 pages
Chapter09_Energiefunktionale.pdf -> 17 pages
合并完成：E:\OneDrive - MSFT\.master_data\26ss\IF\Vorlesungsfolien\_merged\IF_All_Chapters.pdf，总页数：487


## 2. Merge exercise material by exercise number

In [4]:
exercise_sources = [
    (0, "Folien", EXERCISE_SLIDES_DIR),
    (1, "Aufgabe", EXERCISE_TASKS_DIR),
    (2, "Lösung", EXERCISE_SOLUTIONS_DIR),
]
exercise_groups = defaultdict(list)
for role, label, folder in exercise_sources:
    for path in folder.glob("*.pdf"):
        match = re.search(r"(?:übung|ubung)[_ ]*(\d+)", path.stem, re.IGNORECASE)
        if match:
            exercise_groups[int(match.group(1))].append((role, label, path))

EXERCISE_OUTPUT_DIR.mkdir(exist_ok=True)
for number in sorted(exercise_groups):
    entries = sorted(exercise_groups[number], key=lambda item: (item[0], item[2].name))
    files = [item[2] for item in entries]
    titles = [item[1] for item in entries]
    output_path = EXERCISE_OUTPUT_DIR / f"EX{number:02d}.pdf"
    pages, total = merge_pdf_files(files, output_path, titles)
    print(f"{output_path.name}: {len(pages)} files, {total} pages")


EX01.pdf: 3 files, 37 pages
EX02.pdf: 3 files, 28 pages
EX03.pdf: 3 files, 31 pages
EX04.pdf: 3 files, 43 pages
EX05.pdf: 3 files, 36 pages
EX06.pdf: 3 files, 28 pages
EX07.pdf: 2 files, 11 pages
EX08.pdf: 2 files, 6 pages


## 3. Merge all exams with Aufgabe bookmarks

In [5]:
def semester_metadata(path):
    text = path.stem.lower()
    match = re.search(r"(ws|ss)[_ ]?(\d{2,4})(?:[_-]?(\d{2}))?", text)
    if not match:
        return 9999, 9, "Unknown"
    term, digits, second = match.groups()
    if len(digits) == 4 and int(digits[2:]) == int(digits[:2]) + 1:
        year = 2000 + int(digits[:2])
        period = f"{term.upper()}{digits[:2]}/{digits[2:]}"
    elif len(digits) == 4:
        year = int(digits)
        period = f"{term.upper()}{digits}"
    else:
        year = 2000 + int(digits)
        period = f"{term.upper()}{digits}" + (f"/{second}" if second else "")
    term_order = 0 if term == "ss" else 1
    return year, term_order, period

def exam_kind(path):
    first_text = PdfReader(str(path)).pages[0].extract_text() or ""
    return "Solution" if re.search(r"Musterl.sung", first_text, re.IGNORECASE) else "Task"

def exam_sort_key(path):
    year, term_order, period = semester_metadata(path)
    kind_order = 1 if exam_kind(path) == "Solution" else 0
    return year, term_order, kind_order, path.name.lower()

AUFGABE_HEADING = re.compile(r"^Aufgabe\s+(\d+)(?![\d.])\s*:?[ \t]*(.*)$", re.IGNORECASE)

def extract_aufgabe_bookmarks(path):
    reader = PdfReader(str(path))
    bookmarks = []
    seen = set()
    for page_index, page in enumerate(reader.pages):
        lines = [" ".join(line.split()) for line in (page.extract_text() or "").splitlines() if line.strip()]
        page_entries = []
        for line in lines:
            line = re.sub(r"\bA\s+ufgabe\b", "Aufgabe", line, flags=re.IGNORECASE)
            match = AUFGABE_HEADING.match(line)
            if match and match.group(1) not in seen:
                page_entries.append((match.group(1), line))
        if page_index < 3 and len({number for number, _ in page_entries}) > 1:
            continue
        for number, title in page_entries:
            if number not in seen:
                seen.add(number)
                title = re.sub(r"\s*\(\d+(?:[,.]\d+)?\s*Punkte?\)\s*$", "", title, flags=re.IGNORECASE)
                bookmarks.append((number, title, page_index))
    return bookmarks

exam_files = sorted([
    path for path in EXAM_DIR.glob("*.pdf")
    if path.is_file() and "_merged" not in path.parts
], key=exam_sort_key)
if not exam_files:
    raise FileNotFoundError(f"没有找到考试 PDF：{EXAM_DIR}")

writer = PdfWriter()
total_pages = 0
for path in exam_files:
    page_start = total_pages
    page_count = pdf_pages(path)
    writer.append(str(path), import_outline=False)
    _, _, period = semester_metadata(path)
    kind = exam_kind(path)
    exam_node = writer.add_outline_item(f"{period} - {kind}", page_number=page_start)
    bookmarks = extract_aufgabe_bookmarks(path)
    for _, title, page_index in bookmarks:
        writer.add_outline_item(title, page_number=page_start + page_index, parent=exam_node)
    total_pages += page_count
    print(f"{path.name}: {page_count} pages, {len(bookmarks)} Aufgabe bookmarks")

exam_output = EXAM_OUTPUT_DIR / "IF_Exams.pdf"
exam_output.parent.mkdir(parents=True, exist_ok=True)
with exam_output.open("wb") as output:
    writer.write(output)
print(f"合并完成：{exam_output}，文件数：{len(exam_files)}，总页数：{total_pages}")


170213 if ws1617.pdf: 15 pages, 6 Aufgabe bookmarks
180809 if ws1718.pdf: 14 pages, 5 Aufgabe bookmarks
180913 if ss18.pdf: 16 pages, 5 Aufgabe bookmarks
190215 if ws1819.pdf: 15 pages, 5 Aufgabe bookmarks
200206 if ss19.pdf: 17 pages, 5 Aufgabe bookmarks
200310 if ws1920.pdf: 17 pages, 5 Aufgabe bookmarks
200903 if ss20.pdf: 18 pages, 5 Aufgabe bookmarks
IF_ws20_21.pdf: 22 pages, 5 Aufgabe bookmarks
IF_ss21.pdf: 21 pages, 5 Aufgabe bookmarks
IF_ws21_22.pdf: 23 pages, 5 Aufgabe bookmarks
IF_ss2022.pdf: 25 pages, 5 Aufgabe bookmarks
IF_ws22_23.pdf: 21 pages, 5 Aufgabe bookmarks
IF_ss2023.pdf: 23 pages, 5 Aufgabe bookmarks
IF_ss23.pdf: 19 pages, 5 Aufgabe bookmarks
IF_ws23_24.pdf: 24 pages, 5 Aufgabe bookmarks
IF_ss24.pdf: 21 pages, 5 Aufgabe bookmarks
IF_ws24_25.pdf: 25 pages, 5 Aufgabe bookmarks
IF_ss25.pdf: 21 pages, 5 Aufgabe bookmarks
合并完成：E:\OneDrive - MSFT\.master_data\26ss\IF\Weiteres Lernmaterial\Altklausuren_IF_bis_ss25\_merged\IF_Exams.pdf，文件数：18，总页数：357


## 4. Translate all merged PDFs (run separately after merging)

In [6]:
from pathlib import Path
import subprocess
import tempfile
import shutil

FOLDERS = [LECTURE_OUTPUT_DIR, EXERCISE_OUTPUT_DIR, EXAM_OUTPUT_DIR]
OVERWRITE = False
pdf2zh_next_cmd = shutil.which("pdf2zh_next")
if pdf2zh_next_cmd is None:
    raise RuntimeError("没有找到 pdf2zh_next。请先运行：pip install pdf2zh-next")

pdf_files = sorted([
    path for folder in FOLDERS for path in folder.glob("*.pdf")
    if path.is_file()
    and not path.name.startswith("trans-")
    and not path.name.startswith("~$")
])
print("=" * 80)
print(f"待翻译文件夹：{FOLDERS}")
print(f"待翻译 PDF 数量：{len(pdf_files)}")
print("=" * 80)

for pdf_path in pdf_files:
    out_path = pdf_path.with_name("trans-" + pdf_path.name)
    if out_path.exists() and not OVERWRITE:
        print(f"跳过，已存在：{out_path.name}")
        continue
    print("\n" + "-" * 80)
    print(f"开始翻译：{pdf_path.name}")
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        tmp_input = tmpdir / pdf_path.name
        shutil.copy2(pdf_path, tmp_input)
        cmd = [
            pdf2zh_next_cmd,
            str(tmp_input),
            "--lang-in", "de",
            "--lang-out", "zh-CN",
            "--no-mono",
            "--split-short-lines",
            "--short-line-split-factor", "2.0",
            "--ignore-cache",
            "--watermark-output-mode", "no_watermark",
        ]
        result = subprocess.run(cmd, cwd=tmpdir, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(result.stdout[-2000:])
        if result.returncode != 0:
            print(f"失败：{pdf_path.name}")
            continue
        candidates = list(tmpdir.glob(f"{pdf_path.stem}*dual*.pdf"))
        if not candidates:
            candidates = [path for path in tmpdir.glob("*.pdf") if path.name != pdf_path.name]
        if not candidates:
            print(f"没有找到翻译输出文件：{pdf_path.name}")
            continue
        translated_pdf = candidates[0]
        if out_path.exists() and OVERWRITE:
            out_path.unlink()
        shutil.move(str(translated_pdf), str(out_path))
        print(f"完成：{out_path.name}")
print("\n" + "=" * 80)
print("全部处理完成")
print("=" * 80)


待翻译文件夹：[WindowsPath('E:/OneDrive - MSFT/.master_data/26ss/IF/Vorlesungsfolien/_merged'), WindowsPath('E:/OneDrive - MSFT/.master_data/26ss/IF/Uebungsblaetter - merged'), WindowsPath('E:/OneDrive - MSFT/.master_data/26ss/IF/Weiteres Lernmaterial/Altklausuren_IF_bis_ss25/_merged')]
待翻译 PDF 数量：18

--------------------------------------------------------------------------------
开始翻译：EX01.pdf
Parse PDF and Create Intermediate Representation (1/1) ----- 37/37 0:00… 0:00:…
DetectScannedFile (1/1)                                ----- 37/37 0:00… 0:00:…
Parse Page Layout (1/1)                                ----- 74/74 0:00… 0:00:…
Parse Paragraphs (1/1)                                 ----- 37/37 0:00… 0:00:…
Parse Formulas and Styles (1/1)                        ----- 37/37 0:00… 0:00:…
Automatic Term Extraction (1/1)                        ----- 599/… 0:00… 0:00:…
Translate Paragraphs (1/1)                             ----- 599/… 0:00… 0:00:…
Typesetting (1/1)                                